In [1]:
# v3_03_personas, Cell 1: setup on CPU (personas + a few fold-ins need no GPU)
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import drive
from sklearn.cluster import KMeans
from torch import nn

drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/tastelet/data/processed_v3")
MOVIES_CSV = Path("/content/drive/MyDrive/tastelet/data/raw/ml-32m/movies.csv")
MODEL_PATH = DATA_DIR / "ncf_model_v3.pt"

SEED = 42
LIKED_RATING = 4.0
FOLD_IN_STEPS, FOLD_IN_LR, FOLD_IN_NEGATIVES, FOLD_IN_L2 = 100, 0.05, 20, 1e-3  # backend/app/core/config.py
PICK_RATING, NEGATIVE_RATING, POPULARITY_EXPONENT = 5.0, 0.5, 0.75  # ADR 0001 + 0002
FAN_CHUNK = 2_000_000  # embedding lookups in chunks to keep RAM low
PERSONA_K, N_TOP_GENRES, N_TOP_MOVIES = 6, 3, 5
MIN_GENRE_LIKES = 30  # as notebook 04: ignore tiny genres whose lift is noise
MIN_CLUSTER_RATERS = 1_000  # notebook 04 used 10 with ~100 users per cluster; v3 has ~33k
NOT_A_GENRE = {"IMAX", "(no genres listed)"}

torch.manual_seed(SEED)
train = pd.read_parquet(DATA_DIR / "ratings_train.parquet", columns=["user_idx", "movie_idx", "rating"])
movie2idx = json.loads((DATA_DIR / "movie2idx.json").read_text())
n_movies = len(movie2idx)


class NCF(nn.Module):  # identical to backend/app/ml/model.py
    def __init__(self, n_users: int, n_movies: int, dim: int, dropout: float) -> None:
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.movie_embedding = nn.Embedding(n_movies, dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def score_vec(self, user_vecs, movie_ids):
        return self.mlp(torch.cat([user_vecs, self.movie_embedding(movie_ids)], dim=1)).squeeze(-1)

    def forward(self, user_ids, movie_ids):
        return self.score_vec(self.user_embedding(user_ids), movie_ids)


saved = torch.load(MODEL_PATH)  # best epoch, saved with CPU tensors
assert saved["n_movies"] == n_movies, "model and movie2idx disagree"
best_model = NCF(saved["n_users"], saved["n_movies"], saved["embedding_dim"], saved["dropout"])
best_model.load_state_dict(saved["state_dict"])
best_model.eval()
best_model.requires_grad_(False)
all_movies = torch.arange(n_movies)

# Movie titles, years and genres by movie index
movie_info = pd.read_csv(MOVIES_CSV)
movie_info["movie_idx"] = movie_info["movieId"].astype(str).map(movie2idx)
movie_info = movie_info.dropna(subset=["movie_idx"]).astype({"movie_idx": int}).set_index("movie_idx").sort_index()
idx_to_title = movie_info["title"].reindex(range(n_movies)).to_numpy()
movie_years = movie_info["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).astype(float).reindex(range(n_movies)).to_numpy()
genre_rows = movie_info["genres"].str.split("|").explode().rename("genre").reset_index()
genre_rows = genre_rows[~genre_rows["genre"].isin(NOT_A_GENRE)]

# Fold-in with fan start (ADR 0006), same as v3_02_train
rating_counts = np.bincount(train["movie_idx"], minlength=n_movies)
negative_weights = rating_counts.astype(float) ** POPULARITY_EXPONENT
mean_user_vec = best_model.user_embedding.weight.mean(dim=0, keepdim=True)
liked_train = train[train["rating"] >= LIKED_RATING]
liked_movies_t = torch.as_tensor(liked_train["movie_idx"].values, dtype=torch.long)
liked_users_t = torch.as_tensor(liked_train["user_idx"].values, dtype=torch.long)
fan_sums = torch.zeros(n_movies, saved["embedding_dim"])
for chunk_start in range(0, len(liked_train), FAN_CHUNK):
    chunk = slice(chunk_start, chunk_start + FAN_CHUNK)
    fan_sums.index_add_(0, liked_movies_t[chunk], best_model.user_embedding(liked_users_t[chunk]))
fan_counts = torch.bincount(liked_movies_t, minlength=n_movies)
fan_centroids = fan_sums / fan_counts.clamp(min=1).unsqueeze(1)
fan_centroids[fan_counts == 0] = mean_user_vec  # nobody liked it in train: fall back to the average user


def fan_start(picks: list[int]) -> torch.Tensor:
    return fan_centroids[picks].mean(dim=0, keepdim=True)


def fold_in_user_vector(picks: list[int], start: torch.Tensor | None = None) -> torch.Tensor:
    """Same as the backend: picks -> 5.0, 20 popularity-sampled other movies -> 0.5, 100 Adam steps from `start`."""
    rng = np.random.default_rng(SEED)
    user_vec = nn.Parameter((mean_user_vec if start is None else start).clone())
    optimizer = torch.optim.Adam([user_vec], lr=FOLD_IN_LR)
    unpicked = np.setdiff1d(np.arange(n_movies), picks)
    probs = negative_weights[unpicked] / negative_weights[unpicked].sum()
    negatives = rng.choice(unpicked, size=FOLD_IN_NEGATIVES, replace=False, p=probs)
    movie_ids = torch.tensor(np.concatenate([picks, negatives]), dtype=torch.long)
    targets = torch.tensor([PICK_RATING] * len(picks) + [NEGATIVE_RATING] * FOLD_IN_NEGATIVES)
    for _ in range(FOLD_IN_STEPS):
        optimizer.zero_grad()
        predicted = best_model.score_vec(user_vec.expand(len(movie_ids), -1), movie_ids)
        loss = ((predicted - targets) ** 2).mean() + FOLD_IN_L2 * (user_vec**2).sum()
        loss.backward()
        optimizer.step()
    return user_vec.detach()


def find_movie_idx(name: str, year: int) -> int:
    matches = [i for i in range(n_movies) if name.lower() in str(idx_to_title[i]).lower() and movie_years[i] == year]
    assert len(matches) == 1, f"{name} ({year}) matched {[idx_to_title[i] for i in matches]}"
    return matches[0]


HAND_MADE_USERS = {
    "Marvel fan": [("Infinity War - Part II", 2019), ("Black Panther", 2017), ("Thor: Ragnarok", 2017),
                   ("Guardians of the Galaxy", 2014), ("Spider-Man: Into the Spider-Verse", 2018)],
    "horror fan": [("Get Out", 2017), ("Hereditary", 2018), ("Conjuring", 2013),
                   ("Conjuring 2", 2016), ("Quiet Place", 2018)],
    "romance fan": [("La La Land", 2016), ("Notebook", 2004), ("Pride & Prejudice", 2005),
                    ("About Time", 2013), ("Before Sunrise", 1995)],
}

# Control: the Marvel fan's pick score must be close to 4.36 (Cell 19 on GPU)
marvel_picks = [find_movie_idx(name, year) for name, year in HAND_MADE_USERS["Marvel fan"]]
marvel_vec = fold_in_user_vector(marvel_picks, start=fan_start(marvel_picks))
with torch.no_grad():
    marvel_score = best_model.score_vec(marvel_vec.expand(len(marvel_picks), -1), torch.tensor(marvel_picks)).mean().item()
print(f"{len(train):,} train ratings · {n_movies:,} movies · Marvel fan pick score {marvel_score:.2f} (GPU: 4.36)")

Mounted at /content/drive
25,458,081 train ratings · 23,344 movies · Marvel fan pick score 4.36 (GPU: 4.36)


In [2]:
# v3_03_personas, Cell 2: does K = 6 still fit v3? Per K: look-alike pairs, smallest cluster, hand-made users
user_vecs = best_model.user_embedding.weight.numpy()
unit_user_vecs = user_vecs / np.linalg.norm(user_vecs, axis=1, keepdims=True)
genre_names = sorted(genre_rows["genre"].unique())
genre_matrix = np.zeros((n_movies, len(genre_names)))
genre_matrix[genre_rows["movie_idx"].values, pd.Categorical(genre_rows["genre"], categories=genre_names).codes] = 1
liked_user_idx, liked_movie_idx = liked_train["user_idx"].values, liked_train["movie_idx"].values


def cluster_likes(labels: np.ndarray, k: int) -> np.ndarray:
    """Liked ratings per (cluster, movie), shape (k, n_movies)."""
    return np.bincount(labels[liked_user_idx] * n_movies + liked_movie_idx, minlength=k * n_movies).reshape(k, n_movies)


def genre_lift_of(likes: np.ndarray) -> np.ndarray:
    genre_likes = likes @ genre_matrix
    lift = (genre_likes / genre_likes.sum(axis=1, keepdims=True)) / (genre_likes.sum(axis=0) / genre_likes.sum())
    return np.where(genre_likes >= MIN_GENRE_LIKES, lift, 0.0)


hand_made_unit_vecs = {}
for user, named_picks in HAND_MADE_USERS.items():
    picks = [find_movie_idx(name, year) for name, year in named_picks]
    vec = fold_in_user_vector(picks, start=fan_start(picks)).numpy().ravel()
    hand_made_unit_vecs[user] = vec / np.linalg.norm(vec)

kmeans_by_k = {}
for k in range(6, 11):
    kmeans_by_k[k] = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(unit_user_vecs)
    labels = kmeans_by_k[k].labels_
    top3 = [set(np.argsort(-row)[:N_TOP_GENRES]) for row in genre_lift_of(cluster_likes(labels, k))]
    look_alikes = sum(len(top3[a] & top3[b]) >= 2 for a in range(k) for b in range(a + 1, k))
    centroids = kmeans_by_k[k].cluster_centers_ / np.linalg.norm(kmeans_by_k[k].cluster_centers_, axis=1, keepdims=True)
    landed = [int(np.argmax(centroids @ v)) for v in hand_made_unit_vecs.values()]
    print(f"K {k}: look-alike pairs {look_alikes} · smallest cluster {np.bincount(labels).min():,} · "
          f"Marvel/horror/romance -> {landed}")

K 6: look-alike pairs 1 · smallest cluster 27,187 · Marvel/horror/romance -> [0, 2, 2]
K 7: look-alike pairs 0 · smallest cluster 23,535 · Marvel/horror/romance -> [5, 0, 5]
K 8: look-alike pairs 2 · smallest cluster 16,886 · Marvel/horror/romance -> [6, 2, 6]
K 9: look-alike pairs 2 · smallest cluster 15,386 · Marvel/horror/romance -> [8, 4, 8]
K 10: look-alike pairs 4 · smallest cluster 14,555 · Marvel/horror/romance -> [9, 5, 9]


In [3]:
# Cell 3: profile K = 7: top genres and top movies, both ranked by lift; where the hand-made users land
def profile(k: int) -> None:
    labels = kmeans_by_k[k].labels_
    likes = cluster_likes(labels, k)
    lift = genre_lift_of(likes)
    with np.errstate(divide="ignore", invalid="ignore"):  # movies nobody liked: masked out below
        movie_lift = (likes / likes.sum(axis=1, keepdims=True)) / (likes.sum(axis=0) / likes.sum())
    movie_lift = np.where(likes >= MIN_CLUSTER_RATERS, movie_lift, 0.0)  # at least 1,000 likes inside the cluster
    centroids = kmeans_by_k[k].cluster_centers_ / np.linalg.norm(kmeans_by_k[k].cluster_centers_, axis=1, keepdims=True)
    landed = {user: int(np.argmax(centroids @ v)) for user, v in hand_made_unit_vecs.items()}
    for c in range(k):
        genres = {genre_names[g]: round(float(lift[c, g]), 2) for g in np.argsort(-lift[c])[:N_TOP_GENRES]}
        movies = [idx_to_title[m] for m in np.argsort(-movie_lift[c])[:N_TOP_MOVIES]]
        users = [user for user, cluster in landed.items() if cluster == c]
        print(f"\ncluster {c} ({np.bincount(labels)[c]:,} users){'  <- ' + ', '.join(users) if users else ''}: {genres}")
        print("  " + " | ".join(movies))


profile(7)


cluster 0 (23,535 users)  <- horror fan: {'Documentary': 1.5, 'Horror': 1.29, 'Animation': 1.23}
  Holiday, The (2006) | 13 Going on 30 (2004) | Proposal, The (2009) | Enchanted (2007) | How to Lose a Guy in 10 Days (2003)

cluster 1 (25,385 users): {'Romance': 1.47, 'Comedy': 1.27, 'Drama': 1.15}
  Grumpier Old Men (1995) | Phenomenon (1996) | River Wild, The (1994) | Time to Kill, A (1996) | Truth About Cats & Dogs, The (1996)

cluster 2 (31,679 users): {'Horror': 1.29, 'Sci-Fi': 1.25, 'Action': 1.24}
  Star Trek: Insurrection (1998) | Star Trek VI: The Undiscovered Country (1991) | Lord of the Rings, The (1978) | Star Trek III: The Search for Spock (1984) | Star Trek: The Motion Picture (1979)

cluster 3 (27,968 users): {'Film-Noir': 2.62, 'Documentary': 1.87, 'Musical': 1.51}
  Atlantic City (1980) | Nashville (1975) | Klute (1971) | Diva (1981) | Year of Living Dangerously, The (1982)

cluster 4 (27,092 users): {'Crime': 1.43, 'Mystery': 1.38, 'Film-Noir': 1.26}
  There Will Be B

In [4]:
# Cell 4: save the K = 7 personas in the backend's format (centroids.json / personas.json), as _v3 files on Drive
PERSONA_K = 7  # ADR 0007
PERSONA_NAMES = {
    0: ("The Rom-Com Regular", "Feel-good love stories and big-hearted comedies you could rewatch any night."),
    1: ("The Comfort Watcher", "Warm, easy comedies and dramas that feel like a cosy night in."),
    2: ("The Starship Captain", "Star Trek, space missions and classic sci-fi: you boldly go."),
    3: ("The Film Buff", "Arthouse gems, film noir and the classics critics still talk about."),
    4: ("The Case Cracker", "Gritty crime and slow-burn thrillers: the darker the story, the better."),
    5: ("The Blockbuster Hero", "Superheroes, big franchises and epic adventures made for the biggest screen."),
    6: ("The Action Junkie", "Thrillers and action: hackers, hostages and heroes racing the clock."),
}
persona_kmeans = kmeans_by_k[PERSONA_K]
unit_centroids = persona_kmeans.cluster_centers_ / np.linalg.norm(persona_kmeans.cluster_centers_, axis=1, keepdims=True)
persona_genre_lift = genre_lift_of(cluster_likes(persona_kmeans.labels_, PERSONA_K))
personas = [
    {"id": c, "name": name, "description": description,
     "top_genres": [genre_names[g] for g in np.argsort(-persona_genre_lift[c])[:N_TOP_GENRES]]}
    for c, (name, description) in PERSONA_NAMES.items()
]
(DATA_DIR / "centroids_v3.json").write_text(json.dumps(unit_centroids.round(6).tolist()))
(DATA_DIR / "personas_v3.json").write_text(json.dumps(personas, indent=2))

# Check, like the app: nearest normalized centroid for each hand-made user
for user, unit_vec in hand_made_unit_vecs.items():
    print(f"{user:12} -> {PERSONA_NAMES[int(np.argmax(unit_centroids @ unit_vec))][0]}")
print("saved:", sorted(p.name for p in DATA_DIR.glob("*_v3.json")))

Marvel fan   -> The Blockbuster Hero
horror fan   -> The Rom-Com Regular
romance fan  -> The Blockbuster Hero
saved: ['centroids_v3.json', 'personas_v3.json']


In [5]:
# Cell 5: export the small v3 artifacts the backend needs (step 7 copies them into backend/artifacts/)
(DATA_DIR / "movie_rating_counts_v3.json").write_text(json.dumps(rating_counts.tolist()))  # popularity (ADR 0005)
np.save(DATA_DIR / "fan_centroids_v3.npy", fan_centroids.numpy().astype(np.float32))      # fan start (ADR 0006)
for path in sorted(DATA_DIR.glob("*_v3*")):
    print(f"{path.name}: {path.stat().st_size / 1e6:.1f} MB")

centroids_v3.json: 0.0 MB
fan_centroids_v3.npy: 3.0 MB
movie_rating_counts_v3.json: 0.1 MB
ncf_model_v3.pt: 28.7 MB
personas_v3.json: 0.0 MB
